In [ ]:
ENV["OMP_NUM_THREADS"] = "4"
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

isCuda = try
    success(`nvidia-smi`)
catch
    false
end
if isCuda
    println("CUDA is available, using GPU acceleration.")
    using CUDA
end

using bslLD, Random, FFTW, DSP, CairoMakie, Statistics
bslLD.greet()

if isCuda
    println("Setting backend to CUDA.")
    bslLD.use_cuda!()
else
    println("CUDA not available, using CPU.")
end

In [ ]:


function stepStrang!(f, grid, simTime, diag)
    phase_start = simTime.phase
    Ω = simTime.gyro_frequency

    # V half-step at phase(t)
    sol = bslLD.solve_fields(bslLD.Moments(bslLD.compute_density(f, grid)), grid, bslLD.PoissonSolver(-0.001))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f, grid, simTime, sol.E)

    # X full-step at phase(t + dt/2)
    simTime.phase = phase_start + Ω * simTime.dt * 0.5
    simTime.fraction_dt = 1.0
    bslLD.advectX!(f, grid, simTime)

    rho = bslLD.compute_density(f, grid)

    # V half-step at phase(t + dt)
    simTime.phase = phase_start + Ω * simTime.dt
    sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.AdiabaticSolver() )

    bslLD.add_kappaT!(f,grid,simTime.dt,0.1,sol.E)

    simTime.fraction_dt = 0.5
    bslLD.advectV!(f, grid, simTime, sol.E)
    simTime.fraction_dt = 1.0

    # restore so advance!() applies the correct full-step increment
    simTime.phase = phase_start

    diags!(diag, f, rho, sol.E[1], grid, simTime)
end

In [ ]:
Lx = 2pi/0.8
Ly = Lx
Lz = 240*pi

vmax = 4.0

Nx = 16
Ny = Nx
Nz = 8
Nv = 16

nDiag = 1

grid =  bslLD.Grid([0.0,0.0,0.0,-vmax,-vmax,-vmax],[Lx, Ly, Lz, vmax, vmax, vmax],[Nx, Ny, Nz, Nv, Nv, Nv],3, 1.0, 3)
simTime = bslLD.SimulationTime(0.1, 100.0, gyro_frequency=1.0)


initFuncx(x) = 1+ 0.00001 * randn()
f = bslLD.Distribution(grid, 0.0000001, initFuncx = initFuncx);


mutable struct Diag
    rho::Vector
end
Diag() = Diag([])

function diags!(diags, f, rho, Ex, grid, simTime)
    simTime.step % nDiag == 0 || return
    push!(diags.rho, copy(rho.data))
end

In [ ]:
rho = bslLD.compute_density(f, grid)

# V half-step at phase(t + dt)
sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.AdiabaticSolver() )

bslLD.add_kappaT!(f,grid,simTime.dt,0.1,sol.E)


In [ ]:
rho0 = Array(bslLD.compute_density(f,grid).data)

fig = Figure()
ax = Axis(fig[1, 1])
hm = heatmap!(ax, rho0[:,:,1])
Colorbar(fig[1, 2], hm)
fig

In [ ]:
fig = Figure(size = (1200, 400))

fxy = Array(f.data[1,1,1,:,:,8])
fyz = Array(f.data[1,1,1,8,:,:])
fxz = Array(f.data[1,1,1,:,8,:])

ax1 = Axis(fig[2, 1])
hm1 = heatmap!(ax1, fxy)
Colorbar(fig[1, 1], hm1, vertical = false)

ax2 = Axis(fig[2, 2])
hm2 = heatmap!(ax2, fyz)
Colorbar(fig[1, 2], hm2, vertical = false)

ax3 = Axis(fig[2, 3])
hm3 = heatmap!(ax3, fxz)
Colorbar(fig[1, 3], hm3, vertical = false)

fig

In [ ]:
bslLD.ProgressMeter.ijulia_behavior(:clear)

diags = Diag()
while bslLD.continue_advection(simTime, true)
    stepStrang!(f, grid, simTime, diags)
    bslLD.advance!(simTime)
end

In [ ]:
rho = map(Array,diags.rho);

In [ ]:
using Statistics, CairoMakie

In [ ]:
drho2 = map(x->mean((x.-mean(x)).^2),rho);

In [ ]:
fig = Figure(size = (800,400))

ax = Axis(fig[1,1])
heatmap!(ax,rho[1][:,:,1])

ax = Axis(fig[1,2])
heatmap!(ax,rho[end][:,:,1])

fig



In [ ]:
fig = Figure()

ax = Axis(fig[1,1]; yscale = log10)

lines!(ax,collect(simTime)[2:nDiag:end],drho2)

fig